# Multi-period Optimal Power Flow (MPOPF)
---



The multiperiod ACOPF model from potpourri is imported to set up and solve the multiperiod OPF problem.



In [ ]:
import simbench as sb
import pyomo.environ as pe
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
from potpourri.models_multi_period.ACOPF_multi_period import ACOPF_multi_period

1. Load network data

A low-voltage rural SimBench network model is used in this example.


In [ ]:
net = sb.get_simbench_net("1-LV-urban6--0-sw")


2. Set up optimization horizon

The optimization horizon spans from a starting time (fromT) to an ending time (toT). In this case, we set the horizon to 96 time steps (1 day) with 15-minute intervals. The standard timeseries data used for load and generation elements in the grid are the Simbench profiles for each grid element over the defined time horizon.

In [14]:
fromT = 0
toT = fromT + 96 # 1 day for 15 min intervals

3. Customize Generator Power Limits

We then set the power limits for each generator (max_p_mw and min_p_mw) and mark them as controllable. If these limits are not explicitly set, the default max. values provided by SimBench are used.

In [15]:
net.sgen["max_p_mw"] = net.sgen["p_mw"]
net.sgen["min_p_mw"] = net.sgen["p_mw"]
net.sgen['controllable'] = True


4. Set up ACOPF Model

Now, we instantiate the multiperiod ACOPF model from potpourri and add the optimization problem. An optional input for the instance of the OPF is the number of electric vehicles 'num_vehicles' that can offer additional flexibility by optimizing their charging behaviour while taking their mobility needs into account. Currently, the optimization of up to 100 EVs is allowed. To take into account is that the addition of EVs increases the computation time of the optimization problem significantly.
The objective function can be set seperately. In this case, we are minimizing the voltage deviation.

In [16]:
opf = ACOPF_multi_period(net, toT, fromT, num_vehicles=5)
opf.add_OPF()
opf.add_voltage_deviation_objective()

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)



Creating Model at:  Tue Jul 22 13:58:30 2025


4. Solve Multi-period OPF

We solve the ACOPF optimization problem using the ipopt solver. The solver adjusts the power generation and reactive power over the entire time horizon to minimize the objective function while satisfying the grid constraints. After solving the OPF problem, we access some results, such as the power generation of an exemplary pv generation over the entire optimization horizon.


In [17]:
opf.solve(print_solver_output=False)

# example of how to access the results -> sg output throughout the time horizon
for t in opf.model.T:
    print(pe.value(opf.model.psG[0, t]))


Solved successfully
0.0
0.0
2.2086183420621212e-23
2.168229683570399e-20
0.0
0.0
0.0
2.1417990946982502e-23
2.091311109771421e-20
0.0
2.2883169091906498e-20
0.0
2.172935500392635e-20
2.0716731369476798e-20
1.9108093745208525e-19
2.6184662222359103e-20
2.3139283694930723e-20
1.2095733737258836e-21
1.7013617586688563e-23
3.12132855033833e-20
2.2132467871274895e-20
2.1558137830513685e-20
2.5513208011937246e-20
1.5927704558964168e-20
2.040407168060015e-20
0.0
9.658278660340477e-21
3.6949485549832835e-21
8.912823627202082e-24
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
2.6242670390687937e-22
0.0
5.67246562580816e-23
1.6930338110511602e-22
2.993515393486667e-22
2.0528346580349145e-22
2.0401075719654208e-23
3.537797698934654e-24
0.0
1.5890071104147705e-23
0.0
0.0
1.42537380820258e-23
2.868064394851033e-23
4.995264062458875e-24
3.1809190541791757e-23
1.2520971448024824e-23
0.0
1.3505361133126612e-22
0.0
0.0
0.0
0.0
1.209789568648315e-22
2.870370683614297e-23
2.9604794963005434e-23
1.0057128741495491e-

Conclusion

In this notebook, we have demonstrated how to use the potpourri package to solve a Multi-Period Optimal Power Flow (MP-ACOPF) problem. We started by loading network data from Simbench, set up the multiperiod ACOPF optimization horizon and optimization problem, solved it using Pyomo, and showed an example of accessing the results. This workflow can be extended to other power system models and optimization objectives.

